# Plate Zoom в Google Colab

Распознавание номеров с цифровым зумом прямо из браузера (работает и с телефона).

**Перед началом:** меню **Среда выполнения → Сменить среду выполнения → T4 GPU**.

Дальше нажимай ▶ на каждой ячейке по порядку.

## 1. Установка (1–2 минуты)

In [ ]:
!git clone -q https://github.com/aw1rn/camera-kz
%cd camera-kz/plate-zoom
!pip install -q ultralytics easyocr lap
print('Готово')

## 2. Источник видео

Выбери **один** вариант:
- **Видео с телефона** — запусти ячейку и выбери файл (сними дорогу так, чтобы номера были видны).
- **Ссылка на поток** — вставь адрес `.m3u8` в `SOURCE`.

In [ ]:
# Вариант А: загрузить видео с телефона
from google.colab import files
up = files.upload()
SOURCE = list(up.keys())[0]
print('Источник:', SOURCE)

In [ ]:
# Вариант Б: ссылка на поток (пропусти, если загрузил видео)
SOURCE = "https://...m3u8"  # <- вставь ссылку
print('Источник:', SOURCE)

## 3. Запуск

`--max-frames 1500` — примерно минута видео. Для потока это обязательно, иначе он будет идти бесконечно.

In [ ]:
!python main.py --source "$SOURCE" --no-show --save result.mp4 --device 0 --max-frames 1500

## 4. Результаты: найденные номера

In [ ]:
import sqlite3, pandas as pd
from IPython.display import display, Image
con = sqlite3.connect('data/plates.db')
df = pd.read_sql('SELECT plate, conf, votes, ts, image FROM plates ORDER BY id DESC', con)
display(df[['plate', 'conf', 'votes', 'ts']])
for _, r in df.head(10).iterrows():
    print(r.plate)
    display(Image(r.image, width=300))

## 5. Видео с разметкой

In [ ]:
!ffmpeg -y -loglevel error -i result.mp4 -vcodec libx264 -crf 28 -preset veryfast result_web.mp4
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open('result_web.mp4', 'rb').read()).decode()
HTML(f'<video width=100% controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

In [ ]:
# Скачать видео на телефон
from google.colab import files
files.download('result_web.mp4')